# Комфортный маршрут на городском велосипеде

Постройте маршруты между заданными точками Констанца. Меняйте ячейки с ✏️ и выполняйте их сверху вниз.

In [ ]:
from pathlib import Path
import sys

import networkx as nx
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from cycle_routing.workshop import (
    build_network, comfort_cost, data_dictionary, example_rows, export_solution, gpx_table,
    load_city, ml_feature, points_dictionary, preview_cost, preview_features, score, solution, train_custom_model,
)
from cycle_routing.tags import has_any
from cycle_routing.routing import edge_bearings
from cycle_routing.workshop_geo import OSMUnavailableError, city_from_addresses, within_radius
from cycle_routing.workshop_maps import plot_pairs, plot_popularity, plot_turn_examples, route_explorer

PACK = "konstanz"  # набор данных
city = load_city(ROOT / "notebooks/data/hackathon" / PACK)
network = build_network(city)
OUTPUT = ROOT / "outputs/hackathon" / PACK

## 1. Данные

`city.roads` — дороги, `city.trips` — открытые измерения поездок, `city.pairs` — пары A–B.
`NaN` — нет измерения, `0` — измеренный ноль.

In [ ]:
display(data_dictionary(city))

In [ ]:
display(example_rows(city))

In [ ]:
plot_popularity(city);

### Точки

Наборы сохранены локально. Название из таблицы — ключ в `points`.

In [ ]:
points = city.points
display(points_dictionary(city))

## 2. Пары A–B

In [ ]:
plot_pairs(city);

## 3. Настройки ✏️

Стоимость — ощущаемая длина. Признак близости равен 1, если точка попадает в заданный радиус от дороги.

In [ ]:
def features(roads):
    f = pd.DataFrame(index=roads.index)
    f["длина_м"] = roads["длина_м"]  # множитель стоимости, отдельный вес не нужен
    # has_any учитывает значения, разделённые «;».
    f["велодорожка"] = has_any(roads["тип_дороги"], "велодорожка").astype(float)
    f["крупная_дорога"] = has_any(
        roads["тип_дороги"], {"главная_дорога", "дорога_второго_значения", "скоростная_дорога"}
    ).astype(float)
    f["асфальт"] = has_any(roads["покрытие"], "асфальт").astype(float)
    f["светофор_50м"] = within_radius(roads, points["светофоры"], 50).astype(int)
    f["парковка_100м"] = within_radius(roads, points["велопарковки"], 100).astype(int)
    return f

preview_features(city, features)

### Веса ✏️

Плюс — избегать, минус — предпочитать, 0 — не учитывать. Стоимость = длина × (1 + сумма «вес × признак»).

In [ ]:
WEIGHTS = {
    "крупная_дорога": 0.5,
    "велодорожка": -0.25,
    "асфальт": -0.1,
    "светофор_50м": 0.0,
    "парковка_100м": 0.0,
}
my_cost = comfort_cost(WEIGHTS, features=features)
preview_cost(city, my_cost)

### Своя формула ✏️

Функция возвращает ощущаемую длину каждого участка. Для применения выберите `my_formula` в «Сборке решения».

In [ ]:
def my_formula(roads):
    f = features(roads)
    length = f["длина_м"]

    # Признаки равны 0 или 1: поправка работает там, где признак есть.
    factor = 1.0 + 0.5 * f["крупная_дорога"]  # +50% от длины
    factor = factor - 0.25 * f["велодорожка"]  # -25% от длины
    factor = factor - 0.1 * f["асфальт"]       # -10% от длины

    # Множитель не ниже 0.1, чтобы стоимость оставалась положительной.
    cost = length * factor.clip(lower=0.1)
    return cost  # ощущаемые метры: 100 м только с асфальтом → 90 м

preview_cost(city, my_formula, features=features)

### Повороты ✏️

Штрафы — в метрах. Ниже — пример; числа можно менять.

In [ ]:
def my_turn_penalty(graph, incoming, outgoing):
    # incoming и outgoing — направленные участки (откуда, куда, ключ).
    incoming_direction = edge_bearings(graph, incoming)[1]  # конец въезда
    outgoing_direction = edge_bearings(graph, outgoing)[0]  # начало выезда
    # Угол от −180° до 180°: плюс — направо, минус — налево.
    angle = (outgoing_direction - incoming_direction + 180) % 360 - 180
    if abs(angle) <= 25:
        return 0.0  # прямо
    if abs(angle) <= 55:
        return 5.0  # плавный изгиб
    if abs(angle) >= 150:
        return 50.0  # разворот
    if angle > 0:
        return 10.0  # направо
    return 20.0  # налево

In [ ]:
plot_turn_examples(network, my_turn_penalty);  # синий — въезд, зелёный — выезд

### Поиск ✏️

Дейкстра минимизирует стоимость пути.

In [ ]:
def my_search(graph, source, target, weight="cost"):
    return nx.dijkstra_path(graph, source, target, weight=weight)

Жадный поиск выбирает путь ближе к цели по прямой, без учёта стоимости дорог.

In [ ]:
def greedy_search(graph, source, target, weight="cost"):
    if source not in graph or target not in graph:
        raise nx.NodeNotFound("Начало или конец отсутствует в графе")

    # Квадрат расстояния до цели: корень не меняет порядок близости.
    def distance(node):
        # При учёте поворотов вершина — участок дороги, координаты — его конец.
        point, goal = graph.nodes[node], graph.nodes[target]
        return (point["x"] - goal["x"])**2 + (point["y"] - goal["y"])**2

    paths = [[source]]
    visited = {source}  # каждую вершину посещаем один раз

    while paths:
        # Выбираем путь, последняя вершина которого ближе всего к цели.
        path = paths[0]
        for candidate in paths:
            if distance(candidate[-1]) < distance(path[-1]):
                path = candidate
        paths.remove(path)
        current = path[-1]

        if current == target:
            return path

        for neighbor in graph.successors(current):
            if neighbor not in visited:
                visited.add(neighbor)
                paths.append(path + [neighbor])

    raise nx.NetworkXNoPath("Путь не найден")

## 4. Машинное обучение ✏️

`x` — признаки дорог, `y` — `log(1 + число поездок)`.
Начальная модель всегда предсказывает среднее. Замените её своей.

In [ ]:
from sklearn.dummy import DummyRegressor

def train_model(x, y):
    # x — признаки, y — log(1 + поездки); верните обученную модель.
    model = DummyRegressor(strategy="mean")
    model.fit(x, y)
    return model

# Проверка на отложенных кварталах, затем обучение на всех известных поездках.
my_model = train_custom_model(city, train_model, features=features)

def features_with_my_model(roads):
    f = features(roads)
    f["ml_популярность"] = ml_feature(my_model)(roads)["ml_популярность"]
    return f

ML_WEIGHTS = WEIGHTS.copy()
ML_WEIGHTS["ml_популярность"] = -0.25
my_ml_cost = comfort_cost(ML_WEIGHTS, features=features_with_my_model)

## 5. Сборка решения ✏️

Выберите стоимость, повороты и поиск.

In [ ]:
TURNS = my_turn_penalty  # или None
SEARCH = my_search  # или greedy_search

mine = solution(my_cost, turns=TURNS, search=SEARCH, label="моё решение")  # или my_formula
with_my_ml = solution(my_ml_cost, turns=TURNS, search=SEARCH, label="моё + своя ML")
solutions = {
    mine["label"]: mine,
    with_my_ml["label"]: with_my_ml,
}

## 6. Сравнение

Время поиска — медиана на один маршрут, в мс. Добавьте свою метрику — например, число поворотов на километр.

In [ ]:
result = score(city, solutions, network=network)
comparison = result.comparison

routes = result.route_table
routes["поворотов_на_км"] = routes["поворотов"] / (routes["длина_м"] / 1000)
comparison["поворотов на км"] = routes.groupby("решение")["поворотов_на_км"].mean().round(1)
display(comparison)

## 7. Карта

Выберите признак в меню «Цвет». Точки включаются в «Слоях», радиус от них — в «Радиус от точек».

In [ ]:
route_explorer(city, result, features=features, filename=OUTPUT / "routes.html")

## 8. Другой город

### Петербург

Красные линии — GPX. Совпадение считается в пределах 30 м от трека.

In [ ]:
SPB = ROOT / "notebooks/data/hackathon/saint_petersburg"
spb = load_city(SPB)
points = spb.points
try:
    spb_result = score(spb, solutions)
    display(gpx_table(spb_result, spb.tracks))
    display(route_explorer(spb, spb_result, tracks=spb.tracks, features=features,
                           filename=OUTPUT / "spb_routes.html"))
finally:
    points = city.points

### Свои адреса ✏️

Укажите полные адреса с городом. OSM загрузится автоматически; нужен интернет.

In [ ]:
RUN_ADDRESSES = False
START = "Санкт-Петербург, Дворцовая площадь, 2"
END = "Санкт-Петербург, Лиговский проспект, 30"

if RUN_ADDRESSES:
    try:
        local_city = city_from_addresses(START, END, folder=OUTPUT / "addresses")
    except OSMUnavailableError as error:
        print(f"Пример со своими адресами пропущен: {error} Остальные разделы можно выполнять.")
    else:
        points = local_city.points
        try:
            local_result = score(local_city, {mine["label"]: mine})
            display(route_explorer(local_city, local_result, features=features,
                                   filename=OUTPUT / "my_routes.html"))
        finally:
            points = city.points

## 9. Сдача ✏️

Выберите `mine` или `with_my_ml`.

In [ ]:
CHOSEN = mine  # выбранное вами решение
folder = export_solution(city, CHOSEN, folder=OUTPUT / "submission")
print("Сдайте папку", folder, "и этот ноутбук")